# 02 - Data Preprocessing

## Objective

The objective of this notebook is to clean and transform the raw dataset into a suitable format for machine learning model development.

The preprocessing steps include handling missing values, removing irrelevant features, encoding categorical variables, treating outliers where necessary, and scaling numerical features.

## Import Libraries

In [36]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [37]:
df = pd.read_csv("../data/raw/student_dropout_dataset.csv")

df.head()

,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,4,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


## Create a Working Copy

A copy of the raw dataset is created so that the original dataset remains unchanged throughout the preprocessing process.

In [38]:
data = df.copy()

data.shape

(10000, 19)

## Check Missing Values

In [39]:
missing = data.isnull().sum()

missing[missing > 0]

Family_Income          500
Study_Hours_per_Day    500
Stress_Index           500
Parental_Education     511
dtype: int64

## Handling Missing Numerical Values

Missing values in `Family_Income`, `Study_Hours_per_Day`, and `Stress_Index` are handled using median imputation.

The median was selected because it is less affected by extreme values and is therefore suitable for numerical features that may contain skewed distributions.

In [40]:
numerical_imputation_columns = [
    "Family_Income",
    "Study_Hours_per_Day",
    "Stress_Index"
]

for col in numerical_imputation_columns:
    data[col] = data[col].fillna(data[col].median())

## Handling Missing Categorical Values

Missing values in `Parental_Education` are replaced using the mode, which represents the most frequently occurring category in the feature.

In [41]:
data["Parental_Education"] = data["Parental_Education"].fillna(
    data["Parental_Education"].mode()[0]
)

In [42]:
data.isnull().sum().sum()

np.int64(0)

## Removing Irrelevant Features

`Student_ID` is a unique identifier and does not provide meaningful information for predicting student dropout. Therefore, it is removed from the dataset before model training.

In [43]:
data = data.drop(columns=["Student_ID"])

In [44]:
data.head()

,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,24.4,Male,29740.5,Yes,4.00,82.2,2,38.6,No,No,5.5,1.78,1.77,1.77,Year 1,CS,High School,1
4,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


## Separating Features and Target

The dataset is divided into:

* `X` — input features used by the machine learning model
* `y` — target variable representing whether a student drops out

The target variable is `Dropout`.

In [45]:
X = data.drop(columns=["Dropout"])
y = data["Dropout"]

In [46]:
print("Features:", X.shape)
print("Target:", y.shape)

Features: (10000, 17)
Target: (10000,)


## Train-Test Split

The dataset is divided into training and testing sets using an 80:20 ratio.

Stratified splitting is used to maintain a similar distribution of the `Dropout` target classes in both sets.

In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [48]:
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (8000, 17)
Testing set: (2000, 17)


## Encoding Categorical Features

Machine learning algorithms require numerical input. Therefore, categorical features are converted into numerical representations using one-hot encoding.

`drop_first=True` is used to avoid redundant dummy variables for binary and multi-category features.

In [49]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns

categorical_columns

Index(['Gender', 'Internet_Access', 'Part_Time_Job', 'Scholarship', 'Semester',
       'Department', 'Parental_Education'],
      dtype='object')

In [50]:
X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    drop_first=True
)

In [51]:
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

In [52]:
X_train.head()

,Age,Family_Income,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Stress_Index,GPA,Semester_GPA,CGPA,...,Semester_Year 2,Semester_Year 3,Semester_Year 4,Department_Business,Department_CS,Department_Engineering,Department_Science,Parental_Education_High School,Parental_Education_Master,Parental_Education_PhD
4565,19.7,30092.0,4.73,72.1,3,7.7,9.7,0.35,0.68,0.81,...,False,False,True,False,False,True,False,False,True,False
3663,21.7,71435.0,3.46,92.2,0,42.0,8.8,1.21,0.98,0.95,...,False,True,False,False,True,False,False,True,False,False
8964,22.5,53704.0,1.56,70.3,4,30.0,4.7,1.32,1.51,1.50,...,False,False,True,False,False,False,True,True,False,False
5778,21.1,31543.0,3.65,86.2,1,43.6,6.6,2.16,2.37,2.41,...,False,True,False,False,False,False,False,False,True,False
9938,18.2,25000.0,4.54,64.6,3,26.9,6.3,0.44,0.37,0.37,...,False,False,False,True,False,False,False,True,False,False


## Numerical Feature Scaling

The numerical features have different ranges, so StandardScaler is applied to standardize their values.

The scaler is fitted only on the training data and then applied to both the training and testing sets. This prevents information from the testing set from leaking into the training process.

In [53]:
numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

numerical_columns

Index(['Age', 'Family_Income', 'Study_Hours_per_Day', 'Attendance_Rate',
       'Assignment_Delay_Days', 'Travel_Time_Minutes', 'Stress_Index', 'GPA',
       'Semester_GPA', 'CGPA'],
      dtype='object')

In [54]:
scaler = StandardScaler()

X_train[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

In [55]:
X_train[numerical_columns].head()

,Age,Family_Income,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Stress_Index,GPA,Semester_GPA,CGPA
4565,-0.613065,-0.387662,0.564092,-1.175868,0.902618,-1.889146,2.415356,-1.844696,-1.508605,-1.387893
3663,0.321690,1.667159,-0.439646,1.271377,-1.331355,0.979204,1.895382,-1.034976,-1.229282,-1.257381
8964,0.695592,0.785896,-1.941302,-1.395024,1.647276,-0.024300,-0.473385,-0.931408,-0.735811,-0.744657
5778,0.041264,-0.315545,-0.289481,0.540856,-0.586697,1.113004,0.624336,-0.140519,0.064916,0.103668
9938,-1.314131,-0.640744,0.413926,-2.089019,0.902618,-0.283539,0.451012,-1.759958,-1.797239,-1.798072


In [56]:
X_test[numerical_columns].head()

,Age,Family_Income,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Stress_Index,GPA,Semester_GPA,CGPA
7574,-0.753278,3.590072,2.018327,-0.165314,0.157960,1.848907,-1.513332,-0.027535,0.158024,0.159602
3523,1.677084,2.204087,-0.645136,-0.287067,-0.586697,0.870491,1.086535,-0.046366,-0.363380,-0.362445
7890,0.181477,-0.640744,-1.277412,-0.274892,0.157960,1.447506,-0.877809,0.622123,0.707359,0.709615
6115,-1.080442,-0.264750,0.208437,-1.005413,0.902618,0.753415,0.219913,-0.149935,-0.298204,-0.297189
8319,-0.519589,-0.640744,0.010850,0.674785,1.647276,-1.186693,1.548733,-0.865500,-0.856851,-0.856524


## Check Final Data

In [57]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nMissing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

X_train shape: (8000, 24)
X_test shape: (2000, 24)

Missing values in X_train: 0
Missing values in X_test: 0


## Check Target Distribution

In [58]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training target distribution:
Dropout
0    6117
1    1883
Name: count, dtype: int64

Testing target distribution:
Dropout
0    1529
1     471
Name: count, dtype: int64


In [59]:
print(y_train.value_counts(normalize=True) * 100)

Dropout
0    76.4625
1    23.5375
Name: proportion, dtype: float64


## Save cleaned dataset for further analysis

In [63]:
import os

os.makedirs("../data/processed", exist_ok=True)

cleaned_data = data.copy()

cleaned_data.to_csv(
    "../data/processed/student_dropout_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [61]:
cleaned_data.shape

(10000, 18)

In [62]:
cleaned_data.head()

,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,24.4,Male,29740.5,Yes,4.00,82.2,2,38.6,No,No,5.5,1.78,1.77,1.77,Year 1,CS,High School,1
4,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


In [64]:
cleaned_data.isnull().sum().sum()

np.int64(0)

## Preprocessing Summary

The following preprocessing steps were completed:

1. Missing numerical values were handled using median imputation.
2. Missing categorical values were handled using mode imputation.
3. The irrelevant `Student_ID` feature was removed.
4. Categorical features were converted using one-hot encoding.
5. Numerical features were standardized using `StandardScaler`.
6. The dataset was divided into training and testing sets using stratified 80:20 splitting.

The resulting datasets are now ready for exploratory analysis, feature engineering, and machine learning model development.